# Sentiment Analysis of Customer Reviews

### Customer Experience Churn Analytics — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of customer experience and churn analytics terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the NLP task in the context of customer experience and churn analytics and how text features are extracted.
- Implement a text classification pipeline and evaluate accuracy on representative banking queries.
- Evaluate robustness to varied phrasing and identify failure modes relevant to customer-facing deployment.

**Estimated time:** 35–45 minutes

**Collection context:** Customer churn prediction, segmentation, lifetime value, and retention strategy in retail banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Banks receive thousands of reviews and support tickets. Reading them all is impossible. AI can automatically "read" these and flag negative ones for urgent attention, helping to stop churn before it happens.

**Input data used:** Textual customer reviews (e.g., "The app is great!", "Terrible service at the branch.").

**What we predict:** Sentiment label (`positive`, `negative`).

**ML method used:** TF-IDF Vectorization + Logistic Regression.

**Learning goal:** Learn the basics of turning text into numbers for machine learning.

**Primary stakeholders:** relationship managers, retention teams, marketing analysts, and branch managers.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Review Dataset

We'll create 500 positive and 500 negative review snippets.

In [ ]:
pos_reviews = [
    "Great app, very easy to use", "Love the new mobile banking interface",
    "The staff at the branch were very helpful", "Fastest mortgage approval ever",
    "I highly recommend this bank to everyone", "Excellent customer service and rates",
    "Very satisfied with my credit card rewards", "The online chat is very responsive",
    "Best banking experience in years", "Very professional and polite team"
] * 50

neg_reviews = [
    "Terrible service, waited for an hour", "The app keeps crashing on my phone",
    "Very high fees for international transfers", "They blocked my card for no reason",
    "Customer support was very rude and unhelpful", "The branch is always understaffed",
    "Hidden charges everywhere, I'm switching banks", "Takes forever to get a response",
    "Poor communication regarding my loan", "The website is very slow and buggy"
] * 50

reviews = pos_reviews + neg_reviews
labels = [1] * 500 + [0] * 500 # 1=Positive, 0=Negative

df = pd.DataFrame({'review': reviews, 'sentiment': labels})
df = df.sample(frac=1).reset_index(drop=True) # Shuffle

print("Sample reviews:")
print(df.head())

## Step 4 — Exploratory Data Analysis

For this workflow, primary exploration happens through the operational visualisations in later steps.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

For this workflow, the data prepared in Step 3 is used directly as input features.

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the most frequent class ---
from collections import Counter
target_col = df.columns[-1]
majority_class, majority_n = Counter(df[target_col]).most_common(1)[0]
print(f"Majority-class baseline: always predict '{majority_class}' -> accuracy {majority_n/len(df):.3f}")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

Computers don't understand words. TF-IDF turns words into numbers based on how important they are in the document.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df['review'], df['sentiment'], test_size=0.2, random_state=RANDOM_SEED)

vectorizer = TfidfVectorizer(stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)} words")

In [ ]:
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))

In [ ]:
words = vectorizer.get_feature_names_out()
coefs = model.coef_[0]

word_impact = pd.DataFrame({'word': words, 'impact': coefs}).sort_values(by='impact')

plt.figure(figsize=(10, 8))
plt.subplot(1, 2, 1)
sns.barplot(x='impact', y='word', data=word_impact.head(10))
plt.title('Top Negative Words')

plt.subplot(1, 2, 2)
sns.barplot(x='impact', y='word', data=word_impact.tail(10))
plt.title('Top Positive Words')

plt.tight_layout()
plt.show()

**Alt-text:** Bar chart titled "Top Negative Words" and "Top Positive Words". The chart ranks features or categories by magnitude to highlight dominant factors.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

In [ ]:
test_review = ["I am so angry, the branch manager was terrible to me today"]
test_vec = vectorizer.transform(test_review)
prediction = model.predict(test_vec)[0]
prob = model.predict_proba(test_vec)[0]

sentiment = "Positive" if prediction == 1 else "Negative"
print(f"Review: '{test_review[0]}'")
print(f"Predicted Sentiment: {sentiment} ({prob[prediction]:.2%} confidence)")

## Summary
**Banking Context:** By automatically processing reviews, a bank can identify themes (e.g., "the app crashes") and fix them quickly. It can also route "terrible service" complaints directly to a senior manager for immediate recovery.

Let's test the model on a new, unseen review.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: classify new queries ---
print("Operational demonstration — real-time intent classification:\n")
sample_queries = [
    "Can you show me my account balance?",
    "I need to transfer money to savings",
    "My card was stolen, please help",
    "What interest rates do you offer?",
    "I want to close my account",
]
for q in sample_queries:
    intent = model.predict([q])[0]
    print(f'  "{q}"  ->  {intent}')

print("\nIn production, predicted intents would route queries to the correct service channel.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end customer experience and churn analytics workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Churn models that rely on demographic features risk discriminatory retention offers.
- Aggressive retention tactics driven by model outputs may erode customer trust.
- Customers should benefit from personalisation, not be manipulated by it.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in customer experience and churn analytics?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.